# GAT Model Training Notebook

**CBSA – Continuous Behavioural Streaming API**

This notebook is **fully self-contained** and replicates the exact behaviour of the
`POST /train` endpoint:

1. **Installs** all required Python packages
2. **Health check** – verifies Cosmos DB and Azure Blob Storage connections
3. **Loads** behavioural training data from the Cosmos DB `behaviour-logs` container
   (same as the live API)
4. **Trains** the GAT with **CUDA** when available, falls back to **CPU** otherwise,
   ensuring tensors and model always share the same device
5. **Uploads** the trained `.pt` checkpoint to Azure Blob Storage
6. **Marks** every successfully-trained user as `enrolled = true` in the Cosmos
   `enrollment-state` container

> **Configuration**: fill in the credentials in *Cell 2 – Configuration* before running.


In [ ]:
# ── Cell 1: Install Required Packages ───────────────────────────────────────
import subprocess, sys

def pip_install(*packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

# Core ML – installs PyTorch with CUDA 12.1 support.
# If your GPU uses a different CUDA version (e.g. 11.8) change 'cu121'
# to the matching tag, or replace '--index-url .../cu121' with
# '--index-url .../cpu' for a CPU-only build.
pip_install('torch', 'torchvision', 'torchaudio', '--index-url',
            'https://download.pytorch.org/whl/cu121')
pip_install('torch_geometric')
pip_install('torch_scatter', 'torch_sparse', 'torch_cluster', 'torch_spline_conv',
            '-f', 'https://data.pyg.org/whl/torch-2.1.0+cu121.html')

# Azure SDKs
pip_install('azure-cosmos>=4.5.0')
pip_install('azure-storage-blob>=12.19.0')

# Utilities
pip_install('numpy', 'tqdm')

print('✓ All packages installed')


In [ ]:
# ── Cell 2: Configuration ────────────────────────────────────────────────────
import os

# ---- Azure Cosmos DB --------------------------------------------------------
COSMOS_ENDPOINT   = os.environ.get('COSMOS_ENDPOINT', '')          # Required
COSMOS_KEY        = os.environ.get('COSMOS_KEY', '')               # Required
COSMOS_DATABASE   = os.environ.get('COSMOS_DATABASE', 'cbsa-logs') # Default matches app

# Container names (must match app/config.py defaults)
COSMOS_BEHAVIOUR_LOGS_CONTAINER = os.environ.get(
    'COSMOS_BEHAVIOUR_LOGS_CONTAINER', 'behaviour-logs')
COSMOS_PROFILES_CONTAINER = os.environ.get(
    'COSMOS_PROFILES_CONTAINER', 'user-profiles')
COSMOS_ENROLLMENT_CONTAINER = os.environ.get(
    'COSMOS_ENROLLMENT_CONTAINER', 'enrollment-state')

# ---- Azure Blob Storage (model checkpoints) ---------------------------------
AZURE_STORAGE_CONNECTION_STRING = os.environ.get(
    'AZURE_STORAGE_CONNECTION_STRING', '')  # Required for blob upload
AZURE_STORAGE_CONTAINER = os.environ.get(
    'AZURE_STORAGE_CONTAINER', 'cbsa-models')

# ---- Training hyper-parameters (keep in sync with app/triplet_trainer.py) ---
INPUT_DIM               = 56    # 48 behavioural + 8 event-type embedding
HIDDEN_DIM              = 64
OUTPUT_DIM              = 64
NUM_HEADS               = 4
DROPOUT                 = 0.1
TEMPORAL_DIM            = 16
TRIPLET_MARGIN          = 0.5
LEARNING_RATE           = 0.001
MIN_EVENTS_FOR_SESSION  = 5
WINDOW_SECONDS          = 20.0
WINDOW_STRIDE_SECONDS   = 2.0
N_EPOCHS                = 30
CHECKPOINT_BLOB_NAME    = 'gat_checkpoint.pt'
LOCAL_CHECKPOINT_PATH   = '/tmp/gat_checkpoint.pt'

print('Configuration loaded')
print(f'  Cosmos endpoint  : {COSMOS_ENDPOINT[:40] + "..." if len(COSMOS_ENDPOINT) > 40 else COSMOS_ENDPOINT or "(not set)"}')
print(f'  Blob storage     : {"(set)" if AZURE_STORAGE_CONNECTION_STRING else "(not set)"}')


In [ ]:
# ── Cell 3: Health Check – Cosmos DB & Azure Blob Storage ───────────────────
# This cell verifies that all required connections are healthy BEFORE training.
# If any connection check fails the cell raises an error so you can fix
# credentials before wasting GPU time.

from azure.cosmos import CosmosClient, PartitionKey
from azure.storage.blob import BlobServiceClient

_health_errors = []

# ---- 1. Cosmos DB behaviour-logs container ----------------------------------
print('[health] Checking Cosmos DB behaviour-logs container …')
_cosmos_client = None
_cosmos_db = None
_behaviour_logs_container = None
try:
    if not COSMOS_ENDPOINT or not COSMOS_KEY:
        raise ValueError('COSMOS_ENDPOINT and COSMOS_KEY must be set')
    _cosmos_client = CosmosClient(COSMOS_ENDPOINT, credential=COSMOS_KEY)
    _cosmos_db = _cosmos_client.create_database_if_not_exists(id=COSMOS_DATABASE)
    _behaviour_logs_container = _cosmos_db.create_container_if_not_exists(
        id=COSMOS_BEHAVIOUR_LOGS_CONTAINER,
        partition_key=PartitionKey(path='/userId'),
        offer_throughput=400,
    )
    # Quick read probe (list ≤1 item)
    next(iter(_behaviour_logs_container.query_items(
        query='SELECT TOP 1 c.id FROM c',
        enable_cross_partition_query=True,
    )), None)
    print('[health] ✓ Cosmos behaviour-logs: connected')
except Exception as _e:
    _health_errors.append(f'Cosmos behaviour-logs: {_e}')
    print(f'[health] ✗ Cosmos behaviour-logs: {_e}')

# ---- 2. Cosmos DB user-profiles container -----------------------------------
print('[health] Checking Cosmos DB user-profiles container …')
_profiles_container = None
try:
    if _cosmos_db is None:
        raise RuntimeError('Cosmos DB client not connected (see above)')
    _profiles_container = _cosmos_db.create_container_if_not_exists(
        id=COSMOS_PROFILES_CONTAINER,
        partition_key=PartitionKey(path='/userId'),
        offer_throughput=400,
    )
    next(iter(_profiles_container.query_items(
        query='SELECT TOP 1 c.id FROM c',
        enable_cross_partition_query=True,
    )), None)
    print('[health] ✓ Cosmos user-profiles: connected')
except Exception as _e:
    _health_errors.append(f'Cosmos user-profiles: {_e}')
    print(f'[health] ✗ Cosmos user-profiles: {_e}')

# ---- 3. Cosmos DB enrollment-state container --------------------------------
print('[health] Checking Cosmos DB enrollment-state container …')
_enrollment_container = None
try:
    if _cosmos_db is None:
        raise RuntimeError('Cosmos DB client not connected (see above)')
    _enrollment_container = _cosmos_db.create_container_if_not_exists(
        id=COSMOS_ENROLLMENT_CONTAINER,
        partition_key=PartitionKey(path='/userId'),
        offer_throughput=400,
    )
    next(iter(_enrollment_container.query_items(
        query='SELECT TOP 1 c.id FROM c',
        enable_cross_partition_query=True,
    )), None)
    print('[health] ✓ Cosmos enrollment-state: connected')
except Exception as _e:
    _health_errors.append(f'Cosmos enrollment-state: {_e}')
    print(f'[health] ✗ Cosmos enrollment-state: {_e}')

# ---- 4. Azure Blob Storage --------------------------------------------------
print('[health] Checking Azure Blob Storage …')
_blob_container_client = None
try:
    if not AZURE_STORAGE_CONNECTION_STRING:
        raise ValueError('AZURE_STORAGE_CONNECTION_STRING must be set')
    _blob_svc = BlobServiceClient.from_connection_string(AZURE_STORAGE_CONNECTION_STRING)
    _blob_container_client = _blob_svc.get_container_client(AZURE_STORAGE_CONTAINER)
    try:
        _blob_container_client.create_container()
    except Exception:
        pass  # Container already exists
    # Quick probe
    next(iter(_blob_container_client.list_blobs()), None)
    print(f'[health] ✓ Azure Blob Storage ({AZURE_STORAGE_CONTAINER}): connected')
except Exception as _e:
    _health_errors.append(f'Azure Blob Storage: {_e}')
    print(f'[health] ✗ Azure Blob Storage: {_e}')

# ---- Summary ----------------------------------------------------------------
if _health_errors:
    raise RuntimeError(
        'Health check FAILED – fix the following before continuing:\n  ' +
        '\n  '.join(_health_errors)
    )
print('\n[health] All connections healthy – ready to train.')


In [ ]:
# ── Cell 4: Imports and Device Selection ─────────────────────────────────────
import hashlib
import json
import logging
import random
import time
import uuid
from dataclasses import dataclass, field
from types import SimpleNamespace
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool
from tqdm import tqdm

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
)
log = logging.getLogger('gat_notebook')

# ---- Device: CUDA → CPU fallback (all tensors always on the same device) ----
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    log.info('[device] CUDA available – using GPU: %s', torch.cuda.get_device_name(0))
else:
    DEVICE = torch.device('cpu')
    log.info('[device] CUDA not available – falling back to CPU')

print(f'Training device: {DEVICE}')


In [ ]:
# ── Cell 5: Cosmos DB Helpers – Load Training Data ───────────────────────────
# Mirrors app/behavioral_logger.py  +  app/triplet_trainer._normalize_event
# and _load_user_events so that data loading is identical to the /train endpoint.

def _normalize_event(raw: dict) -> dict:
    """
    Normalize a raw Cosmos event document into the canonical trainer format.

    The behaviour-logs container holds documents written by two code-paths:
      1. behavioral_logger.log_event  → timestamp, event_type, event_data.vector
      2. cosmos_prototype_store       → eventTimestamp, eventType, vectorJson

    Returns a dict that always has 'timestamp', 'event_type', 'event_data.vector'.
    """
    ev = dict(raw)

    # ---- timestamp ----------------------------------------------------------
    if 'timestamp' not in ev or ev['timestamp'] is None:
        ev['timestamp'] = (
            ev.get('eventTimestamp')
            or ev.get('loggedAt')
            or ev.get('logged_at')
            or 0.0
        )
    ev['timestamp'] = float(ev['timestamp'])

    # ---- event_type ---------------------------------------------------------
    if 'event_type' not in ev or ev['event_type'] is None:
        ev['event_type'] = ev.get('eventType', 'unknown')

    # ---- event_data / vector ------------------------------------------------
    if 'event_data' not in ev or not isinstance(ev.get('event_data'), dict):
        vector_json = ev.get('vectorJson')
        vector: List[float] = []
        if vector_json:
            try:
                parsed = json.loads(vector_json) if isinstance(vector_json, str) else vector_json
                vector = [float(v) for v in parsed]
            except (json.JSONDecodeError, ValueError, TypeError):
                pass
        ev['event_data'] = {'vector': vector}

    return ev


def list_cosmos_users() -> List[str]:
    """Return all distinct userIds in the behaviour-logs container."""
    log.info('[load] Listing users from Cosmos DB …')
    try:
        items = list(_behaviour_logs_container.query_items(
            query='SELECT DISTINCT c.userId FROM c',
            enable_cross_partition_query=True,
        ))
        users = [item['userId'] for item in items if item.get('userId')]
        log.info('[load] Found %d users in Cosmos DB', len(users))
        return users
    except Exception as exc:
        log.error('[load] Failed to list users from Cosmos: %s', exc)
        return []


def load_user_events(user_id: str) -> List[dict]:
    """Load, normalise, and sort all events for a user from Cosmos DB."""
    log.info('[load] Fetching events for user \'%s\' from Cosmos DB …', user_id)
    try:
        items = list(_behaviour_logs_container.query_items(
            query='SELECT * FROM c WHERE c.userId = @uid ORDER BY c.loggedAt ASC',
            parameters=[{'name': '@uid', 'value': user_id}],
            partition_key=user_id,
        ))
    except Exception as exc:
        log.error('[load] Failed to fetch events for \'%s\': %s', user_id, exc)
        return []

    if not items:
        log.warning('[load] No events found for user \'%s\'', user_id)
        return []

    log.info('[load] Fetched %d raw events for user \'%s\'', len(items), user_id)
    normalized = [_normalize_event(e) for e in items]
    sorted_events = sorted(normalized, key=lambda e: e.get('timestamp', 0))

    ts_min = sorted_events[0].get('timestamp', 0)
    ts_max = sorted_events[-1].get('timestamp', 0)
    log.info(
        '[load] User \'%s\': %d events, timestamp range %.3f → %.3f (span %.1fs)',
        user_id, len(sorted_events), ts_min, ts_max, ts_max - ts_min,
    )
    return sorted_events


print('Cosmos helpers defined')


In [ ]:
# ── Cell 6: Session Windowing ────────────────────────────────────────────────
# Identical to app/triplet_trainer._split_into_windows

def split_into_windows(
    events: List[dict],
    window_sec: float = WINDOW_SECONDS,
    stride_sec: float = WINDOW_STRIDE_SECONDS,
) -> List[List[dict]]:
    """Split a sorted event list into overlapping time windows."""
    if not events:
        log.warning('[window] No events to window')
        return []

    first_ts = float(events[0].get('timestamp', 0) or 0.0)
    last_ts  = float(events[-1].get('timestamp', 0) or 0.0)
    log.info(
        '[window] Windowing %d events: first_ts=%.3f, last_ts=%.3f, span=%.1fs, '
        'window_sec=%.1f, stride_sec=%.1f',
        len(events), first_ts, last_ts, last_ts - first_ts, window_sec, stride_sec,
    )

    windows = []
    win_start = first_ts
    while win_start <= last_ts:
        win_end = win_start + window_sec
        window = [
            ev for ev in events
            if win_start <= float(ev.get('timestamp', 0) or 0.0) < win_end
        ]
        if len(window) >= MIN_EVENTS_FOR_SESSION:
            windows.append(window)
        win_start += stride_sec

    log.info('[window] Created %d session windows (min_events_per_window=%d)',
             len(windows), MIN_EVENTS_FOR_SESSION)
    return windows


print('Windowing helper defined')


In [ ]:
# ── Cell 7: Data Models (dataclasses, no Pydantic) ───────────────────────────
# Mirrors app/gat/models.py but uses stdlib dataclasses to avoid Pydantic.

@dataclass
class EventNode:
    node_id: int
    timestamp: float
    event_type: str
    behavioral_vector: List[float]
    signature: str = ''
    nonce: str = ''


@dataclass
class TemporalEdge:
    source_node_id: int
    target_node_id: int
    time_delta: float
    event_transition: str
    attention_weight: Optional[float] = None


@dataclass
class TemporalGraph:
    session_id: str
    user_id: Optional[str]
    nodes: List[EventNode]
    edges: List[TemporalEdge]
    window_start: float
    window_end: float
    total_events: int
    session_duration: float
    event_diversity: int
    avg_time_between_events: float


print('Data models defined')


In [ ]:
# ── Cell 8: Data Processor ───────────────────────────────────────────────────
# Mirrors app/gat/data_processor.py (BehavioralDataProcessor + PyTorchDataConverter)

class BehavioralDataProcessor:
    """Converts raw behavioural event lists into TemporalGraph objects."""

    def __init__(self, config: Dict[str, Any]):
        self.time_window            = config.get('time_window_seconds', 20)
        self.min_events             = config.get('min_events_per_window', 5)
        self.max_events             = config.get('max_events_per_window', 100)
        self.distinct_target        = config.get('distinct_event_connections', 4)

    def process_behavioral_data(
        self, raw_data: List[Dict], user_id: str, session_id: str
    ) -> TemporalGraph:
        events = sorted(raw_data, key=lambda x: x.get('timestamp', 0))
        events = self._filter_time_window(events)
        nodes  = self._create_event_nodes(events)
        edges  = self._create_temporal_edges(nodes)
        meta   = self._calculate_metadata(events)
        return TemporalGraph(
            session_id=session_id, user_id=user_id,
            nodes=nodes, edges=edges,
            window_start=meta['window_start'], window_end=meta['window_end'],
            total_events=len(nodes), session_duration=meta['session_duration'],
            event_diversity=int(meta['event_diversity']),
            avg_time_between_events=meta['avg_time_between_events'],
        )

    # ------------------------------------------------------------------
    def _filter_time_window(self, events: List[Dict]) -> List[Dict]:
        if not events:
            return []
        latest = max(e.get('timestamp', 0) for e in events)
        filtered = [e for e in events if e.get('timestamp', 0) >= latest - self.time_window]
        if len(filtered) < self.min_events and len(events) >= self.min_events:
            filtered = events[-self.min_events:]
        return filtered[-self.max_events:]

    def _event_type_embedding(self, event_type: str) -> List[float]:
        digest = hashlib.sha256(str(event_type).encode('utf-8')).digest()
        return [b / 255.0 for b in digest[:8]]

    def _extract_behavioral_vector(self, event: Dict) -> List[float]:
        event_data  = event.get('event_data', {}) or {}
        base_vector = list(event_data.get('vector') or [])
        # Pad / truncate to 48-D and guard against None values
        base_vector = (base_vector + [0.0] * 48)[:48]
        base_vector = [float(v) if v is not None else 0.0 for v in base_vector]
        embedding   = self._event_type_embedding(event.get('event_type', 'unknown'))
        return (base_vector + embedding)[:56]

    def _create_event_nodes(self, events: List[Dict]) -> List[EventNode]:
        nodes = []
        for i, event in enumerate(events):
            nodes.append(EventNode(
                node_id=i,
                timestamp=float(event.get('timestamp', 0)),
                event_type=event.get('event_type', 'unknown'),
                behavioral_vector=self._extract_behavioral_vector(event),
                signature=event.get('event_data', {}).get('signature', ''),
                nonce=event.get('event_data', {}).get('nonce', ''),
            ))
        return nodes

    def _create_temporal_edges(self, nodes: List[EventNode]) -> List[TemporalEdge]:
        edges = []
        distinct_target = max(self.distinct_target, 1)
        for i, src in enumerate(nodes):
            seen_types = {src.event_type}
            for j in range(i + 1, len(nodes)):
                tgt = nodes[j]
                edges.append(TemporalEdge(
                    source_node_id=src.node_id, target_node_id=tgt.node_id,
                    time_delta=tgt.timestamp - src.timestamp,
                    event_transition=f'{src.event_type}->{tgt.event_type}',
                ))
                if tgt.event_type not in seen_types:
                    seen_types.add(tgt.event_type)
                    if len(seen_types) >= distinct_target:
                        break
        return edges

    def _calculate_metadata(self, events: List[Dict]) -> Dict[str, float]:
        if not events:
            return dict(window_start=0.0, window_end=0.0,
                        session_duration=0.0, event_diversity=0,
                        avg_time_between_events=0.0)
        timestamps   = [e.get('timestamp', 0) for e in events]
        event_types  = [e.get('event_type', 'unknown') for e in events]
        ws, we       = min(timestamps), max(timestamps)
        avg_dt = (np.mean([timestamps[i+1] - timestamps[i]
                           for i in range(len(timestamps) - 1)])
                  if len(timestamps) > 1 else 0.0)
        return dict(window_start=float(ws), window_end=float(we),
                    session_duration=float(we - ws),
                    event_diversity=float(len(set(event_types))),
                    avg_time_between_events=float(avg_dt))


class PyTorchDataConverter:
    """Converts a TemporalGraph to the tensor dict expected by GATTrainer."""

    def convert_to_pytorch(
        self, temporal_graph: TemporalGraph, device: torch.device = torch.device('cpu')
    ) -> Dict[str, Any]:
        nodes = temporal_graph.nodes
        edges = temporal_graph.edges

        x = torch.FloatTensor([n.behavioral_vector for n in nodes]).to(device)
        temporal_feats = torch.FloatTensor(
            [[n.timestamp - temporal_graph.window_start] for n in nodes]
        ).to(device)

        if edges:
            edge_index = torch.LongTensor(
                [[e.source_node_id for e in edges],
                 [e.target_node_id for e in edges]]
            ).to(device)
        else:
            n = len(nodes)
            edge_index = torch.LongTensor([list(range(n)), list(range(n))]).to(device)

        return {
            'x': x,
            'edge_index': edge_index,
            'temporal_features': temporal_feats,
            'num_nodes': len(nodes),
            'num_edges': len(edges),
            'batch': None,
        }


print('Data processor defined')


In [ ]:
# ── Cell 9: GAT Network ──────────────────────────────────────────────────────
# Mirrors app/gat/gat_network.py (TemporalGraphAttention, SiameseGATNetwork, GATTrainer)

class TemporalGraphAttention(nn.Module):
    """Multi-head GAT with temporal features."""

    def __init__(self, input_dim=56, hidden_dim=64, output_dim=64,
                 num_heads=4, dropout=0.1, temporal_dim=16,
                 return_attention_weights=True):
        super().__init__()
        self.temporal_dim = temporal_dim
        self.return_attention_weights = return_attention_weights
        self.attention_weights_layer1 = None
        self.attention_weights_layer2 = None

        self.temporal_encoder = nn.Sequential(
            nn.Linear(1, temporal_dim), nn.ReLU(), nn.Dropout(dropout)
        )
        combined_dim = input_dim + temporal_dim
        self.gat_layer1 = GATConv(combined_dim, hidden_dim // num_heads,
                                  heads=num_heads, dropout=dropout, concat=True)
        self.gat_layer2 = GATConv(hidden_dim, output_dim // num_heads,
                                  heads=num_heads, dropout=dropout, concat=True)
        self.global_pool = nn.Sequential(
            nn.Linear(output_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim)
        )
        self.layer_norm1 = nn.LayerNorm(hidden_dim)
        self.layer_norm2 = nn.LayerNorm(output_dim)
        self.dropout     = nn.Dropout(dropout)

    def forward(self, x, edge_index, temporal_features, batch=None,
                return_attention=False):
        temporal_encoded = self.temporal_encoder(temporal_features)
        x_combined = torch.cat([x, temporal_encoded], dim=1)

        if return_attention and self.return_attention_weights:
            x1, attn1 = self.gat_layer1(x_combined, edge_index,
                                        return_attention_weights=True)
            self.attention_weights_layer1 = attn1
        else:
            x1 = self.gat_layer1(x_combined, edge_index)
            attn1 = None

        x1 = self.layer_norm1(x1)
        x1 = F.relu(x1)
        x1 = self.dropout(x1)

        if return_attention and self.return_attention_weights:
            x2, attn2 = self.gat_layer2(x1, edge_index,
                                        return_attention_weights=True)
            self.attention_weights_layer2 = attn2
        else:
            x2 = self.gat_layer2(x1, edge_index)
            attn2 = None

        x2 = self.layer_norm2(x2)
        x2 = F.relu(x2)

        if batch is not None:
            graph_emb = global_mean_pool(x2, batch)
        else:
            graph_emb = torch.mean(x2, dim=0, keepdim=True)
        graph_emb = self.global_pool(graph_emb)

        attn_dict = None
        if return_attention and attn1 is not None and attn2 is not None:
            attn_dict = {'layer1': attn1, 'layer2': attn2}

        return x2, graph_emb.squeeze(0), attn_dict


class SiameseGATNetwork(nn.Module):
    """Siamese GAT network for metric learning with triplet loss."""

    def __init__(self, gat_config: Dict):
        super().__init__()
        self.gat_network = TemporalGraphAttention(**gat_config)
        self.similarity_layer = nn.Sequential(
            nn.Linear(gat_config['output_dim'] * 2, gat_config['hidden_dim']),
            nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(gat_config['hidden_dim'], 1), nn.Sigmoid(),
        )

    def forward_once(self, x, edge_index, temporal_features, batch=None,
                     return_attention=False):
        _, graph_emb, attn = self.gat_network(
            x, edge_index, temporal_features, batch,
            return_attention=return_attention,
        )
        return (graph_emb, attn) if return_attention else graph_emb

    def forward(self, data1, data2):
        emb1 = self.forward_once(data1.x, data1.edge_index,
                                 data1.temporal_features, data1.batch)
        emb2 = self.forward_once(data2.x, data2.edge_index,
                                 data2.temporal_features, data2.batch)
        return F.cosine_similarity(emb1, emb2, dim=-1), emb1, emb2


class GATTrainer:
    """Manages triplet-loss training for SiameseGATNetwork."""

    def __init__(self, model: SiameseGATNetwork, learning_rate=0.001,
                 margin=0.2, device='cpu'):
        self.device        = device
        self.model         = model.to(device)
        self.optimizer     = torch.optim.Adam(model.parameters(), lr=learning_rate)
        self.scheduler     = torch.optim.lr_scheduler.StepLR(
            self.optimizer, step_size=10, gamma=0.9)
        self.margin        = margin
        self.triplet_loss  = nn.TripletMarginLoss(margin=margin)

    def triplet_loss_custom(self, anchor, positive, negative):
        pos_sim = F.cosine_similarity(anchor, positive, dim=-1)
        neg_sim = F.cosine_similarity(anchor, negative, dim=-1)
        return torch.clamp(neg_sim - pos_sim + self.margin, min=0.0).mean()

    def train_batch(self, anchor_data, positive_data, negative_data) -> float:
        self.model.train()
        self.optimizer.zero_grad()
        anc = self.model.forward_once(anchor_data.x, anchor_data.edge_index,
                                      anchor_data.temporal_features, anchor_data.batch)
        pos = self.model.forward_once(positive_data.x, positive_data.edge_index,
                                      positive_data.temporal_features, positive_data.batch)
        neg = self.model.forward_once(negative_data.x, negative_data.edge_index,
                                      negative_data.temporal_features, negative_data.batch)
        loss = self.triplet_loss_custom(anc, pos, neg)
        loss.backward()
        self.optimizer.step()
        return loss.item()


print('GAT network defined')


In [ ]:
# ── Cell 10: Training Loop ───────────────────────────────────────────────────
# Replicates app/triplet_trainer.TripletTrainer._train_gat_all_users exactly.
# All graph tensors are created on DEVICE so model and data share the same device.

dp_config = {
    'time_window_seconds':     WINDOW_SECONDS,
    'min_events_per_window':   MIN_EVENTS_FOR_SESSION,
    'max_events_per_window':   100,
    'distinct_event_connections': 4,
}
processor = BehavioralDataProcessor(dp_config)
converter = PyTorchDataConverter()

# ---- 1. Discover users ------------------------------------------------------
log.info('[train_all] === Starting train_all ===')
users = list_cosmos_users()
if not users:
    raise RuntimeError('[train_all] No users found in Cosmos DB behaviour-logs')
log.info('[train_all] Discovered %d users: %s', len(users), users)

# ---- 2. Build per-user graph data -------------------------------------------
log.info('[gat_train] === Starting GAT triplet training for %d users ===', len(users))
user_graphs: Dict[str, list] = {}
skipped: List[dict] = []

for uid in tqdm(users, desc='Loading graphs'):
    log.info('[gat_train] Loading & windowing events for user \'%s\' …', uid)
    events  = load_user_events(uid)
    windows = split_into_windows(events)

    if len(windows) < 2:
        log.warning('[gat_train] User \'%s\' skipped: %d windows (need >= 2)', uid, len(windows))
        skipped.append({
            'user_id': uid, 'status': 'error',
            'message': f'Need at least 2 session windows, found {len(windows)}',
            'profile_saved': False,
        })
        continue

    graphs = []
    for i, w in enumerate(windows):
        try:
            tg = processor.process_behavioral_data(w, uid, f'{uid}_session_{i}')
            gd_dict = converter.convert_to_pytorch(tg, device=DEVICE)
            graphs.append(SimpleNamespace(
                x=gd_dict['x'],
                edge_index=gd_dict['edge_index'],
                temporal_features=gd_dict['temporal_features'],
                batch=gd_dict['batch'],
            ))
        except Exception as exc:
            log.warning('[gat_train] Failed to convert window %d for \'%s\': %s', i, uid, exc)

    if len(graphs) >= 2:
        user_graphs[uid] = graphs
        log.info('[gat_train] User \'%s\': %d valid graphs from %d windows',
                 uid, len(graphs), len(windows))
    else:
        log.warning('[gat_train] User \'%s\' skipped: only %d valid graphs', uid, len(graphs))
        skipped.append({
            'user_id': uid, 'status': 'error',
            'message': 'Not enough valid session graphs',
            'profile_saved': False,
        })

if not user_graphs:
    raise RuntimeError('[gat_train] No users with sufficient data – aborting')

log.info('[gat_train] %d users ready for training, %d skipped',
         len(user_graphs), len(skipped))

# ---- 3. Initialise model on DEVICE ------------------------------------------
gat_config = {
    'input_dim': INPUT_DIM, 'hidden_dim': HIDDEN_DIM, 'output_dim': OUTPUT_DIM,
    'num_heads': NUM_HEADS, 'dropout': DROPOUT, 'temporal_dim': TEMPORAL_DIM,
}
model   = SiameseGATNetwork(gat_config)
trainer = GATTrainer(model, learning_rate=LEARNING_RATE,
                     margin=TRIPLET_MARGIN, device=DEVICE)
log.info('[gat_train] Model initialised on %s', DEVICE)

# ---- 4. Triplet training loop -----------------------------------------------
all_user_ids = list(user_graphs.keys())
t0 = time.time()
log.info('[gat_train] Starting %d epochs of triplet training on %d users …',
         N_EPOCHS, len(all_user_ids))

for epoch in range(N_EPOCHS):
    epoch_loss = 0.0
    count = 0

    for uid in all_user_ids:
        graphs       = user_graphs[uid]
        other_users  = [u for u in all_user_ids if u != uid]

        for i in range(len(graphs)):
            for j in range(len(graphs)):
                if i == j:
                    continue

                anchor   = graphs[i]
                positive = graphs[j]

                if other_users:
                    neg_uid  = random.choice(other_users)
                    negative = random.choice(user_graphs[neg_uid])
                else:
                    # Single user: synthetic negative via additive Gaussian noise.
                    # Scale 0.5 is large enough to push the embedding away from the
                    # anchor while still producing a plausible behavioural vector.
                    # This fallback only fires when there is exactly one user in the
                    # dataset and is provided for development/testing purposes.
                    noisy_x  = anchor.x + torch.randn_like(anchor.x) * 0.5
                    negative = SimpleNamespace(
                        x=noisy_x.clamp(0.0, 1.0),
                        edge_index=anchor.edge_index,
                        temporal_features=anchor.temporal_features,
                        batch=anchor.batch,
                    )

                loss        = trainer.train_batch(anchor, positive, negative)
                epoch_loss += loss
                count      += 1

    if (epoch + 1) % 5 == 0 or epoch == 0:
        avg_loss = epoch_loss / max(count, 1)
        log.info('[gat_train] Epoch %d/%d – avg triplet loss: %.4f (%d batches)',
                 epoch + 1, N_EPOCHS, avg_loss, count)

elapsed = time.time() - t0
log.info('[gat_train] GAT triplet training complete in %.1fs', elapsed)


In [ ]:
# ── Cell 11: Save Checkpoint Locally and Upload to Azure Blob Storage ────────

import os

# Save locally
os.makedirs(os.path.dirname(LOCAL_CHECKPOINT_PATH), exist_ok=True)
torch.save(model.state_dict(), LOCAL_CHECKPOINT_PATH)
log.info('[checkpoint] Model saved locally → %s', LOCAL_CHECKPOINT_PATH)

# Upload to Azure Blob Storage
if _blob_container_client is not None:
    try:
        blob_client = _blob_container_client.get_blob_client(CHECKPOINT_BLOB_NAME)
        with open(LOCAL_CHECKPOINT_PATH, 'rb') as fh:
            blob_client.upload_blob(fh, overwrite=True)
        log.info('[checkpoint] Model uploaded to blob: %s', CHECKPOINT_BLOB_NAME)
        print(f'✓ Checkpoint uploaded to Azure Blob Storage → {CHECKPOINT_BLOB_NAME}')
    except Exception as exc:
        log.error('[checkpoint] Failed to upload model to blob: %s', exc)
        print(f'✗ Blob upload failed: {exc}')
else:
    log.warning('[checkpoint] Blob storage not configured – checkpoint kept locally only')
    print('⚠ Blob storage not available – checkpoint saved locally only')


In [ ]:
# ── Cell 12: Generate Per-User Profiles and Mark Enrolled ───────────────────
# Mirrors the profile-generation + enrollment-marking done by POST /train.

# Must be kept in sync with ENROLLMENT_DURATION_SECONDS in app/enrollment_store.py.
# The notebook cannot import from the app package, so the value is duplicated here.
ENROLLMENT_DURATION_SECONDS = 5 * 60  # 5 minutes – matches app/enrollment_store.py

log.info('[gat_train] Generating per-user profiles …')
results: List[dict] = list(skipped)

model.eval()
for uid, graphs in tqdm(user_graphs.items(), desc='Generating profiles'):
    # ---- generate 64-D profile vector via mean-pool of session embeddings ----
    embeddings = []
    with torch.no_grad():
        for gd in graphs:
            emb = model.forward_once(
                gd.x, gd.edge_index, gd.temporal_features, gd.batch
            )
            embeddings.append(emb)

    profile_tensor = torch.stack(embeddings).mean(dim=0)
    profile_vector = profile_tensor.cpu().numpy().tolist()

    # ---- save profile to Cosmos DB user-profiles container ------------------
    now = time.time()
    profile_doc = {
        'id':              uid,
        'userId':          uid,
        'profileVector':   profile_vector,
        'vectorDim':       len(profile_vector),
        'method':          'gat_triplet',
        'sessionsUsed':    len(graphs),
        'trainingTimeSec': elapsed,
        'createdAt':       now,
        'updatedAt':       now,
    }
    try:
        _profiles_container.upsert_item(profile_doc)
        log.info('[gat_train] Profile saved to Cosmos for user \'%s\' (%d sessions)',
                 uid, len(graphs))
    except Exception as exc:
        log.error('[gat_train] Failed to save profile for \'%s\': %s', uid, exc)

    # ---- mark user as enrolled in Cosmos DB enrollment-state container ------
    enrollment_doc = {
        'id':                 uid,
        'userId':             uid,
        'user_id':            uid,
        'enrolled':           True,
        'accumulated_seconds': ENROLLMENT_DURATION_SECONDS,
    }
    try:
        _enrollment_container.upsert_item(enrollment_doc)
        log.info('[enroll] User \'%s\' marked as enrolled', uid)
    except Exception as exc:
        log.error('[enroll] Failed to mark \'%s\' as enrolled: %s', uid, exc)

    results.append({
        'user_id':                uid,
        'status':                 'success',
        'message':                f'GAT triplet training complete in {elapsed:.1f}s',
        'profile_saved':          True,
        'sessions_used':          len(graphs),
        'training_time_seconds':  elapsed,
    })

log.info('[gat_train] === GAT training complete: %d profiles saved, %d skipped ===',
         len(user_graphs), len(skipped))

# ---- Final summary ----------------------------------------------------------
successes = [r for r in results if r.get('status') == 'success']
errors    = [r for r in results if r.get('status') == 'error']
print(f'\nTraining complete.')
print(f'  Successful : {len(successes)} users')
print(f'  Skipped    : {len(errors)} users')
print(f'  Total time : {elapsed:.1f}s')
print(f'  Device     : {DEVICE}')
for r in results:
    status_icon = '✓' if r.get('status') == 'success' else '✗'
    print(f'  {status_icon} {r.get("user_id", "?")} – {r.get("message", "")}')
